IBTrACS data processing


In [2]:
import pandas as pd

ibtracs_raw = pd.read_csv(
    '../data/IBTrACS.WP.list.v04r01.csv',
    skiprows=[1],
    na_values=[' ', '']
)

C:\Users\SAM\AppData\Local\Temp\ipykernel_14872\3667492051.py:3: DtypeWarning: Columns (0: WMO_AGENCY, 1: USA_AGENCY, 2: USA_ATCF_ID, 3: USA_STATUS, 4: HKO_CAT, 5: KMA_CAT, 6: NEWDELHI_GRADE, 7: DS824_STAGE) have mixed types. Specify dtype option on import or set low_memory=False.
  ibtracs_raw = pd.read_csv(


In [8]:
cols_needed = [
    'SID', 'SEASON', 'NAME', 'ISO_TIME',
    'LAT', 'LON',
    'WMO_WIND', 'USA_WIND',
    'TRACK_TYPE'
]
ibtracs = ibtracs_raw[cols_needed].copy()

ibtracs.columns = [
    'storm_id', 'season', 'storm_name', 'datetime',
    'lat', 'lon',
    'wmo_wind', 'usa_wind',
    'track_type'
]

ibtracs = ibtracs[ibtracs['track_type'] == 'main']

ibtracs['datetime'] = pd.to_datetime(ibtracs['datetime'], errors='coerce')
ibtracs['season']   = pd.to_numeric(ibtracs['season'],    errors='coerce')
ibtracs['lat']      = pd.to_numeric(ibtracs['lat'],       errors='coerce')
ibtracs['lon']      = pd.to_numeric(ibtracs['lon'],       errors='coerce')
ibtracs['wmo_wind'] = pd.to_numeric(ibtracs['wmo_wind'],  errors='coerce')
ibtracs['usa_wind'] = pd.to_numeric(ibtracs['usa_wind'],  errors='coerce')

ibtracs = ibtracs.dropna(subset=['lat', 'lon', 'datetime'])

ibtracs['usa_wind_converted'] = ibtracs['usa_wind'] * 0.88

ibtracs['wind_speed'] = ibtracs['wmo_wind'].where(
    ibtracs['wmo_wind'].notna(),
    ibtracs['usa_wind_converted']
)

ibtracs = ibtracs.dropna(subset=['wind_speed'])

ibtracs = ibtracs.drop(columns=['wmo_wind', 'usa_wind', 'usa_wind_converted'])

ibtracs['year']  = ibtracs['datetime'].dt.year
ibtracs['month'] = ibtracs['datetime'].dt.month

ibtracs = ibtracs[
    (ibtracs['year'] >= 2000) &
    (ibtracs['year'] <= 2025)
]

ibtracs = ibtracs.drop(columns=['track_type'])

print('Cleaned shape:', ibtracs.shape)
print('Unique storms:', ibtracs['storm_id'].nunique())
print('Years covered:', ibtracs['year'].min(), '–', ibtracs['year'].max())
ibtracs.to_csv('../data_clean/ibtracs_cleaned.csv', index=False)
ibtracs.head()

Cleaned shape: (41034, 9)
Unique storms: 732
Years covered: 2000 – 2024


,storm_id,season,storm_name,datetime,lat,lon,wind_speed,year,month
195528,2000125N06136,2000,DAMREY,2000-05-03 18:00:00,6.2,135.7,22.0,2000,5
195529,2000125N06136,2000,DAMREY,2000-05-03 21:00:00,6.8,135.5,22.0,2000,5
195530,2000125N06136,2000,DAMREY,2000-05-04 00:00:00,7.3,135.4,22.0,2000,5
195531,2000125N06136,2000,DAMREY,2000-05-04 03:00:00,7.7,135.3,22.0,2000,5
195532,2000125N06136,2000,DAMREY,2000-05-04 06:00:00,8.1,135.3,22.0,2000,5
